# 01 — Exploratory Data Analysis (EDA)

**Project:** Deep Learning for Early Detection of Diabetic Retinopathy  
**Goal of this notebook:** understand the data *before* modeling — how many images, how the
grades are distributed, what the images look like, and how we'll split the data fairly.

> ⚠️ **Not a medical device.** Research/education only.

### The ICDR 0–4 grading scale (what we're predicting)
Diabetic retinopathy severity is graded on the International Clinical Diabetic Retinopathy
(ICDR) scale:

| grade | meaning | what it looks like |
|--:|--|--|
| 0 | No DR | healthy retina |
| 1 | Mild | a few microaneurysms (tiny red dots) |
| 2 | Moderate | more microaneurysms, some hemorrhages/exudates |
| 3 | Severe | many hemorrhages across the retina |
| 4 | Proliferative | abnormal new blood vessels (sight-threatening) |

It is an **ordinal** scale: the order matters and the distance matters. Predicting grade 0 when
the truth is grade 4 is a *much* worse error than predicting 0 when the truth is 1. We'll keep
this in mind when choosing our metric later (Quadratic Weighted Kappa).

### Referable DR (the binary view)
A clinically useful simplification is **referable DR = grade ≥ 2**: patients at moderate or worse
should generally be referred to an ophthalmologist. We report this binary framing alongside the
full 5-class view because it maps directly to a real screening decision.


In [ ]:
# --- Setup ---------------------------------------------------------------
import sys, os
sys.path.append(os.path.abspath('..'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns, cv2
from src.utils import load_config, set_seed
from src.data import read_labels, class_distribution, make_splits, generate_synthetic_dataset

cfg = load_config('../config.yaml')
set_seed(cfg['project']['seed'])
CLASS_NAMES = cfg['classes']['names']
NUM_CLASSES = cfg['classes']['num_classes']
FIG = '../reports/figures'; os.makedirs(FIG, exist_ok=True)
print('Classes:', CLASS_NAMES)


### Load the labels
If the real APTOS data isn't present yet, this notebook **falls back to a tiny synthetic demo**
set so you can see the whole pipeline run. Synthetic results are meaningless — they only prove
the code works. Once you drop the real APTOS data into `data/aptos2019/`, re-run and you'll get
the real figures.


In [ ]:
# Try the real data; fall back to a synthetic demo so the notebook always runs.
try:
    df = read_labels(cfg)
    DATA_KIND = 'REAL APTOS 2019'
except FileNotFoundError:
    print('Real data not found — generating a small SYNTHETIC demo set (for testing only).')
    demo = generate_synthetic_dataset('../data/_synthetic_demo/aptos2019', seed=cfg['project']['seed'])
    demo_cfg = {**cfg, 'paths': {**cfg['paths'],
        'train_csv': '../data/_synthetic_demo/aptos2019/train.csv',
        'train_images': '../data/_synthetic_demo/aptos2019/train_images'}}
    df = read_labels(demo_cfg)
    DATA_KIND = 'SYNTHETIC DEMO (not real results)'
print(f'Using: {DATA_KIND} | {len(df)} images')
df.head()


## 1. Class distribution — the imbalance
The single most important fact about this dataset: it is **heavily imbalanced** toward 'No DR'.
On the real APTOS training set the split is roughly:

| grade | count | share |
|--:|--:|--:|
| 0 No DR | 1,805 | 49.3% |
| 1 Mild | 370 | 10.1% |
| 2 Moderate | 999 | 27.3% |
| 3 Severe | 193 | 5.3% |
| 4 Proliferative | 295 | 8.1% |

Nearly half the images are healthy, and the two most severe grades are the rarest. **Why this
matters:** a lazy model could reach ~50% accuracy by always predicting 'No DR' while missing
every sick patient — the most dangerous failure. This is why we (a) use class-weighted loss
and/or a balanced sampler, and (b) never judge the model on accuracy alone.


In [ ]:
counts = class_distribution(df, NUM_CLASSES)
fig, ax = plt.subplots(figsize=(8,4.5))
sns.barplot(x=CLASS_NAMES, y=counts.values, color='#2A9D8F', ax=ax)
for i,v in enumerate(counts.values): ax.text(i, v+0.3, str(v), ha='center')
ax.set_ylabel('number of images'); ax.set_title(f'Class distribution — {DATA_KIND}')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
fig.tight_layout(); fig.savefig(f'{FIG}/eda_class_distribution.png', dpi=140, bbox_inches='tight')
plt.show()

# Referable DR (grade >= 2) binary view
thr = cfg['classes']['referable_threshold']
ref = (df['diagnosis'] >= thr).mean()
print(f'Referable DR (grade >= {thr}): {ref*100:.1f}% of images')


## 2. What do the images look like? (one sample per grade)


In [ ]:
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(15,3.4))
for g in range(NUM_CLASSES):
    sub = df[df.diagnosis==g]
    if len(sub)==0:
        axes[g].axis('off'); axes[g].set_title(f'{CLASS_NAMES[g]}\n(none)'); continue
    img = cv2.cvtColor(cv2.imread(sub.iloc[0]['path']), cv2.COLOR_BGR2RGB)
    axes[g].imshow(img); axes[g].set_title(CLASS_NAMES[g], fontsize=10); axes[g].axis('off')
fig.suptitle(f'Sample image per grade — {DATA_KIND}', fontweight='bold')
fig.tight_layout(); fig.savefig(f'{FIG}/eda_sample_grid.png', dpi=140, bbox_inches='tight'); plt.show()


## 3. Image-quality checks
Real fundus datasets contain off-center, blurred, over/under-exposed, or otherwise poor images.
We scan basic stats: pixel dimensions, aspect ratio, and mean brightness. Very dark images
(mean brightness below a threshold) are flagged as candidates for review — image quality is a
known driver of model error in DR grading.


In [ ]:
widths, heights, bright = [], [], []
for p in df['path']:
    im = cv2.imread(p); h,w = im.shape[:2]; widths.append(w); heights.append(h)
    bright.append(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY).mean())
widths, heights, bright = map(np.array, (widths, heights, bright))
ratios = widths/heights
fig, ax = plt.subplots(1,3, figsize=(15,4))
ax[0].scatter(widths, heights, alpha=.4, color='#2A9D8F'); ax[0].set_title('Image size (W vs H)')
ax[0].set_xlabel('width px'); ax[0].set_ylabel('height px')
ax[1].hist(ratios, bins=15, color='#E9C46A'); ax[1].set_title('Aspect ratio (W/H)')
ax[2].hist(bright, bins=20, color='#264653'); ax[2].axvline(25, color='red', ls='--', label='too-dark (<25)')
ax[2].set_title('Mean brightness'); ax[2].set_xlabel('mean pixel'); ax[2].legend()
fig.tight_layout(); fig.savefig(f'{FIG}/eda_quality_checks.png', dpi=140, bbox_inches='tight'); plt.show()
print(f'Flagged too-dark images: {(bright<25).sum()}  |  size range: {widths.min()}-{widths.max()} x {heights.min()}-{heights.max()}')


## 4. Stratified train / val / test split
We split the data **once**, with a fixed seed, into train / validation / test. The split is
**stratified** by grade so each subset keeps the same class proportions as the whole dataset —
essential here, because a plain random split could leave almost no Severe/Proliferative images
in validation or test, making those metrics unreliable.

- **Train** — the model learns from these.
- **Validation** — used during training to pick the best epoch and tune choices.
- **Test** — touched only once, at the very end, for an honest final score.


In [ ]:
# make_splits needs scikit-learn (in requirements.txt; available on Colab).
try:
    train_df, val_df, test_df = make_splits(df, cfg)
    summary = pd.DataFrame({
        'train': class_distribution(train_df, NUM_CLASSES),
        'val':   class_distribution(val_df, NUM_CLASSES),
        'test':  class_distribution(test_df, NUM_CLASSES),
    })
    summary.index = CLASS_NAMES
    print(f'Sizes -> train {len(train_df)}, val {len(val_df)}, test {len(test_df)}')
    display(summary)
    # proportions should match across splits (that's the point of stratifying)
    print('\nClass share per split (%):')
    display((summary / summary.sum() * 100).round(1))
except ModuleNotFoundError:
    print('scikit-learn not installed here. Run `pip install -r requirements.txt` (or on Colab).')


## Written summary

**Dataset.** APTOS 2019 provides ~3,662 labeled fundus photographs graded 0–4 on the ICDR scale.

**Key finding — imbalance.** ~49% of images are 'No DR' (grade 0), while the two most severe
grades (Severe ~5%, Proliferative ~8%) are rare. Roughly **38% of images are 'referable'**
(grade ≥ 2). This imbalance is the central modeling challenge: we counter it with class-weighted
loss and/or a `WeightedRandomSampler` (see `config.yaml → imbalance`), and we evaluate with
metrics that survive imbalance (macro-F1, per-class recall, and Quadratic Weighted Kappa) rather
than overall accuracy.

**Image quality.** Fundus images vary in size, centering, and exposure. We flag very dark images
and will standardize everything in preprocessing (field-of-view crop, Ben-Graham normalization,
resize) so the model sees consistent inputs.

**Splitting.** A single, seed-fixed, **stratified** 70/15/15 train/val/test split keeps class
proportions identical across subsets, so validation and test scores are trustworthy.

_Next: preprocessing & augmentation, with before/after visuals._
